# Day 3: Baseline ML with Enhanced DIP Features
Trains RF and SVM using the standardized `src.features` module.

In [1]:
import cv2
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from pathlib import Path
from tqdm import tqdm
import sys
sys.path.append('..')
from src.features import extract_lulc_features
from src.utils import update_metrics

DATA_DIR = Path('../data/processed')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
data = []
labels = []
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])

print('Generating Standardized Dataset...')
for i, cls in enumerate(classes):
    paths = list((DATA_DIR / cls).glob('*.jpg'))[:500] # Increased for better accuracy
    for img_path in tqdm(paths, desc=f'Class {cls}'):
        img = cv2.imread(str(img_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            feat = extract_lulc_features(img_rgb)
            data.append(feat)
            labels.append(i)

X = np.array(data)
y = np.array(labels)
print(f'Feature vector size: {X.shape[1]}')

Generating Standardized Dataset...


Class Water: 100%|██████████| 500/500 [00:05<00:00, 84.86it/s]

Feature vector size: 168


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
joblib.dump(rf, MODEL_DIR / 'rf_baseline.joblib')
acc_rf = accuracy_score(y_test, rf.predict(X_test))
update_metrics('DIP Features + RF', round(acc_rf * 100, 2))

print('Training SVM...')
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test))
update_metrics('SVM', round(acc_svm * 100, 2))

print('Training KNN (Advanced Comparison)...')
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test))
update_metrics('K-Nearest Neighbors', round(acc_knn * 100, 2))

print('Training Gradient Boosting (XGBoost alternative)...')
gbc = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbc.fit(X_train, y_train)
acc_gbc = accuracy_score(y_test, gbc.predict(X_test))
update_metrics('Gradient Boosting (XGB)', round(acc_gbc * 100, 2))

print('\nReport (RF Baseline):')
print(classification_report(y_test, rf.predict(X_test), target_names=classes))

Training Random Forest...
Updated DIP Features + RF with 87.2% accuracy.
Training SVM...
Updated SVM with 75.0% accuracy.
Training KNN (Advanced Comparison)...
Updated K-Nearest Neighbors with 78.2% accuracy.
Training Gradient Boosting (XGBoost alternative)...
Updated Gradient Boosting (XGB) with 89.0% accuracy.

Report (RF Baseline):
              precision    recall  f1-score   support

 Agriculture       0.91      0.87      0.89       100
   Buildings       0.92      0.97      0.95       100
      Forest       0.99      0.97      0.98       100
       Roads       0.84      0.67      0.74       100
       Water       0.73      0.88      0.80       100

    accuracy                           0.87       500
   macro avg       0.88      0.87      0.87       500
weighted avg       0.88      0.87      0.87       500

